In [0]:
create schema if not exists 03_gold_catalog.dimensions;
create schema if not exists 03_gold_catalog.facts;

In [0]:
create table if not exists 03_gold_catalog.dimensions.dim_customer(
  customer_sk bigint generated always as identity,
  customer_id int,
  customer_name string,
  country string,
  industry_type string,
  account_created_date date,
  is_active boolean
)
using delta;

merge into 03_gold_catalog.dimensions.dim_customer as target
using (
  select 
    customer_id,
    customer_name,
    country,
    industry_type,
    account_created_date,
    is_active
  from 02_silver_catalog.transformed_schema.customer
) as source
on target.customer_id = source.customer_id
when matched and (
  target.customer_name != source.customer_name
  or target.country != source.country
  or target.industry_type != source.industry_type
  or target.account_created_date != source.account_created_date
  or target.is_active != source.is_active
) then
  update set
    target.customer_name = source.customer_name,
    target.country = source.country,
    target.industry_type = source.industry_type,
    target.account_created_date = source.account_created_date,
    target.is_active = source.is_active
when not matched then
  insert (
    customer_id,
    customer_name,
    country,
    industry_type,
    account_created_date,
    is_active
  ) values (
    source.customer_id,
    source.customer_name,
    source.country,
    source.industry_type,
    source.account_created_date,
    source.is_active
  )

In [0]:
create table if not exists 03_gold_catalog.dimensions.dim_employee(
  employee_sk bigint generated always as identity,
  employee_id int,
  employee_name string,
  role string,
  region string,
  hire_date date,
  last_update date,
  is_active boolean
)
using delta;

merge into 03_gold_catalog.dimensions.dim_employee as target
using (
  select 
    employee_id,
    employee_name,
    role,
    region,
    hire_date,
    last_update,
    is_active
  from 02_silver_catalog.transformed_schema.employee
)as source on target.employee_id = source.employee_id

when matched and (
  target.employee_name != source.employee_name
  or target.role != source.role
  or target.region != source.region
  or target.hire_date != source.hire_date
  or target.last_update != source.last_update
  or target.is_active != source.is_active
)then 
  update set
    target.employee_name = source.employee_name,
    target.role = source.role,
    target.region = source.region,
    target.hire_date = source.hire_date,
    target.last_update = source.last_update,
    target.is_active = source.is_active

when not matched then
  insert (
    employee_id,
    employee_name,
    role,
    region,
    hire_date,
    last_update,
    is_active
  )values(
    source.employee_id,
    source.employee_name,
    source.role,
    source.region,
    source.hire_date,
    source.last_update,
    source.is_active
  )

In [0]:
create table if not exists `03_gold_catalog`.dimensions.dim_product(
  product_sk bigint generated always as identity,
  product_id int,
  product_name string,
  plan_name string,
  billing_cycle string,
  list_price int,
  is_active boolean,
  created_date date
)
using delta;

merge into `03_gold_catalog`.dimensions.dim_product as target
using (
  select 
    product_id,
    product_name,
    plan_name,
    billing_cycle,
    list_price,
    is_active,
    created_date
  from `02_silver_catalog`.transformed_schema.products
)as source on target.product_id = source.product_id

when matched and (
  target.product_name != source.product_name
  or target.plan_name != source.plan_name
  or target.billing_cycle != source.billing_cycle
  or target.list_price != source.list_price
  or target.is_active != source.is_active
  or target.created_date != source.created_date
)then 
  update set
    target.product_name = source.product_name,
    target.plan_name = source.plan_name,
    target.billing_cycle = source.billing_cycle,
    target.list_price = source.list_price,
    target.is_active = source.is_active,
    target.created_date = source.created_date

when not matched then
  insert (
    product_id,
    product_name,
    plan_name,
    billing_cycle,
    list_price,
    is_active,
    created_date
  )values(
    source.product_id,
    source.product_name,
    source.plan_name,
    source.billing_cycle,
    source.list_price,
    source.is_active,
    source.created_date
  )

In [0]:
create table if not exists 03_gold_catalog.facts.fact_opportunity(
  opportunity_sk bigint generated always as identity,
  opportunity_id int,
  customer_sk bigint,
  employee_sk bigint,
  product_sk bigint,
  start_date date,
  end_date date,
  contract_term string,
  revenue_amount double,
  close_status string
)
using delta;

merge into 03_gold_catalog.facts.fact_opportunity as target
using (

  select 
    o.opportunity_id,
    dc.customer_sk,
    de.employee_sk,
    dp.product_sk,
    o.start_date,
    o.end_date,
    o.contract_term,
    o.revenue_amount,
    o.close_status
  from 02_silver_catalog.transformed_schema.opportunity o

  inner join 03_gold_catalog.dimensions.dim_customer dc
    on o.customer_id = dc.customer_id

  inner join 03_gold_catalog.dimensions.dim_employee de
    on o.employee_id = de.employee_id

  inner join 03_gold_catalog.dimensions.dim_product dp
    on o.product_id = dp.product_id

) as source

on target.opportunity_id = source.opportunity_id

when matched and (
  NOT (target.customer_sk <=> source.customer_sk)
  OR NOT (target.employee_sk <=> source.employee_sk)
  OR NOT (target.product_sk <=> source.product_sk)
  OR NOT (target.start_date <=> source.start_date)
  OR NOT (target.end_date <=> source.end_date)
  OR NOT (target.contract_term <=> source.contract_term)
  OR NOT (target.revenue_amount <=> source.revenue_amount)
  OR NOT (target.close_status <=> source.close_status)
)

then update set
  target.customer_sk = source.customer_sk,
  target.employee_sk = source.employee_sk,
  target.product_sk = source.product_sk,
  target.start_date = source.start_date,
  target.end_date = source.end_date,
  target.contract_term = source.contract_term,
  target.revenue_amount = source.revenue_amount,
  target.close_status = source.close_status

when not matched then
  insert (
    opportunity_id,
    customer_sk,
    employee_sk,
    product_sk,
    start_date,
    end_date,
    contract_term,
    revenue_amount,
    close_status
  )
  values (
    source.opportunity_id,
    source.customer_sk,
    source.employee_sk,
    source.product_sk,
    source.start_date,
    source.end_date,
    source.contract_term,
    source.revenue_amount,
    source.close_status
  );

In [0]:
create table if not exists `03_gold_catalog`.dimensions.dim_exchange_rate(
  currency_code string,
  fx_rate_to_gbp double,
  effective_date date
)
using delta;

merge into 03_gold_catalog.dimensions.dim_exchange_rate as target
using (
  select 
    currency_code,
    fx_rate_to_gbp,
    effective_date
  from 02_silver_catalog.transformed_schema.fx_rate
)as source on target.currency_code = source.currency_code
and target.effective_date = source.effective_date

when matched and (
  target.fx_rate_to_gbp != source.fx_rate_to_gbp
)then 
  update set
    target.fx_rate_to_gbp = source.fx_rate_to_gbp

when not matched then
  insert (
    currency_code,
    fx_rate_to_gbp,
    effective_date
  )values(
    source.currency_code,
    source.fx_rate_to_gbp,
    source.effective_date
  )

In [0]:
create table if not exists 03_gold_catalog.dimensions.dim_country_master(
  country_code string,
  country_name string,
  currency_code string
)
using delta;

merge into 03_gold_catalog.dimensions.dim_country_master as target
using (
  select 
    country_code,
    country_name,
    currency_code
  from 02_silver_catalog.transformed_schema.country_master
)as source on target.country_code = source.country_code

when matched and (
  target.country_name != source.country_name
  or target.currency_code != source.currency_code
)then 
  update set
    target.country_name = source.country_name,
    target.currency_code = source.currency_code

when not matched then
  insert (
    country_code,
    country_name,
    currency_code
  )values(
    source.country_code,
    source.country_name,
    source.currency_code
  )
